<center><h1>Last_First_HW3</h1></center>
<br>
<br>

Name: Krish Kalawadia
<br>
Github Username: KK10-USC
<br>
USC ID: 4532974788

## 1. Time Series Classification Part 1: Feature Creation/Extraction

### (a) Download Data

Package imports

In [3]:
import pandas as pd
import numpy as np
from pathlib import Path
import re


Get the AReM Data Set

In [8]:
my_folder = Path("HW_3_Data")
time_series_cols = ["avg_rss12","var_rss12","avg_rss13","var_rss13","vg_rss23","ar_rss23"]

my_files = list(my_folder.rglob("dataset*.csv"))
print(len(my_files))
files_frames = []

for p in my_files:
    new_rows = []
    for l in p.read_text().splitlines():
        if l.strip() and not l.startswith("#"):
            new_rows.append(l.replace(","," ").split())
    temp_df = (pd.DataFrame(new_rows).astype(float))
    temp_df["label"] = p.parent.name
    temp_df["sequence"] = p.name
    files_frames.append(temp_df)

my_df = pd.concat(files_frames,ignore_index=True)

    


88


In [9]:
print(my_df)

              0      1     2      3     4      5     6     label      sequence
0           0.0  39.25  0.43  22.75  0.43  33.75  1.30  bending1  dataset1.csv
1         250.0  39.25  0.43  23.00  0.00  33.00  0.00  bending1  dataset1.csv
2         500.0  39.25  0.43  23.25  0.43  33.00  0.00  bending1  dataset1.csv
3         750.0  39.50  0.50  23.00  0.71  33.00  0.00  bending1  dataset1.csv
4        1000.0  39.50  0.50  24.00  0.00  33.00  0.00  bending1  dataset1.csv
...         ...    ...   ...    ...   ...    ...   ...       ...           ...
42234  118750.0  31.50  1.66  12.50  3.20  14.25  4.44   walking  dataset9.csv
42235  119000.0  27.33  1.25  11.33  0.94  20.00  4.00   walking  dataset9.csv
42236  119250.0  37.80  7.68  14.20  2.48  17.25  0.83   walking  dataset9.csv
42237  119500.0  33.75  1.30  15.75  5.21  16.50  2.69   walking  dataset9.csv
42238  119750.0  32.67  3.09  18.67  0.47  14.00  3.16   walking  dataset9.csv

[42239 rows x 9 columns]


### (b) Test and Train Data

In [10]:
bending_folder = my_df["label"].str.startswith("bending")

bending_testData = my_df["sequence"].isin(["dataset1.csv","dataset2.csv"])
other_testData = my_df["sequence"].isin(["dataset1.csv","dataset2.csv","dataset3.csv"])

my_df["test_splitting"] = (bending_folder & bending_testData) | (~bending_folder & other_testData)

training_data = my_df[~my_df["test_splitting"]]
testing_data = my_df[my_df["test_splitting"]]


### (c) Feature Extraction

#### i. Research

Time-Domain Features to Explore in a Time Series Classification: Mean,Median,Standard Deviation,Variance,Min/Max,Range,IQR,Quartiles 1 & 3

#### ii. Extraction

In [13]:
cols = [1,2,3,4,5,6]

feature_ext = my_df.groupby(["sequence","label"])[cols].agg(["min","max","mean","median",  lambda x: x.quantile(0.25),lambda x: x.quantile(0.75)])

statistics = ["min","max","mean","median","Q1","Q3"]

new_col_names = []
for c in cols:
    for s in statistics:
        new_col_names.append(f"{c}_{s}")

feature_ext.columns = new_col_names

#### iii. Standard Deviation

In [15]:
X_val = feature_ext

bootstrp_list = []
for i in range(1000):
    resampling = X_val.sample(len(X_val),replace = True)
    bootstrp_list.append(resampling.std())

boot_df = pd.DataFrame(bootstrp_list)

my_results = pd.DataFrame({"STD": X_val.std(),"CI_UPPER":boot_df.quantile(0.05),"CI_LOWER":boot_df.quantile(0.95)})

print(my_results)

               STD  CI_UPPER   CI_LOWER
1_min     9.569975  8.312155  10.665002
1_max     4.394362  3.338713   5.286333
1_mean    5.335718  4.684862   5.875646
1_median  5.440054  4.765177   5.995372
1_Q1      6.153590  5.534266   6.629074
1_Q3      5.138925  4.351705   5.840142
2_min     0.000000  0.000000   0.000000
2_max     5.062729  4.638194   5.391148
2_mean    1.574164  1.407235   1.702078
2_median  1.412244  1.248446   1.536622
2_Q1      0.946386  0.836886   1.033175
2_Q3      2.125266  1.905842   2.291847
3_min     2.956462  2.766014   3.107163
3_max     4.875137  4.199852   5.474676
3_mean    4.008380  3.428637   4.474550
3_median  4.036396  3.438898   4.518291
3_Q1      4.220658  3.646981   4.682222
3_Q3      4.171628  3.570948   4.675342
4_min     0.000000  0.000000   0.000000
4_max     2.183625  1.977987   2.362106
4_mean    1.166114  1.081143   1.220757
4_median  1.145586  1.063916   1.201613
4_Q1      0.843620  0.777810   0.888396
4_Q3      1.552504  1.441595   1.624404


#### iv. Select Features

In [23]:
chosen_feats = ["mean","median","max"]

chosen_cols = []
for c in feature_ext.columns:
    s = c.split("_")[-1]
    if s in chosen_feats:
        chosen_cols.append(c)

selected_X = feature_ext[chosen_cols]
print(selected_X.shape)
print(selected_X)

(88, 18)
                       1_max     1_mean  1_median  2_max    2_mean  2_median  \
sequence     label                                                             
dataset1.csv bending1  45.00  40.624792     40.50   1.30  0.358604      0.43   
             bending2  51.00  24.562958     24.25   6.87  0.590833      0.43   
             cycling   45.00  37.177042     36.25   8.58  2.374208      1.92   
             lying     30.00  27.716375     27.50   1.79  0.363687      0.43   
             sitting   48.00  42.363562     43.00   4.44  0.492563      0.50   
...                      ...        ...       ...    ...       ...       ...   
dataset9.csv cycling   44.67  37.144833     36.33  10.76  2.419063      1.88   
             lying     56.25  47.325125     42.67   8.49  0.274312      0.00   
             sitting   46.50  43.190854     42.50   1.50  0.266271      0.43   
             standing  47.67  45.399625     45.50   1.66  0.460146      0.50   
             walking   43.67  3

## 2. ISLR 3.7.4

### (a) Linear Train

I would expect when comparing the training RSS of a linear regression model and a cubic regression model that the training RSS for the cubic regression model would be worse than the RSS of a linear regression model. This is because the cubic regression model can iterate over the data and analyze the data better than a linear regression model. A cubic regression model also has higher flexbility than a linear model, which allows the cubic model to fit the data more accurate compared to the linear model. 

### (b) Linear Test

When comparing the testing RSS of a cubic regression model and a linear regression model, it can be implied that the linear regression model would have a lower testing RSS than a cubic regression model. This is because of the bias-variance trade off where if there is no bias and high variance, the cubic regression model would overfit the data.

### (c) Not Linear Train

When the relationship between X and Y is not linear, the model which has a lower RSS compared to the other is the training RSS of a cubic regression model. The relationship of variables does not affect the cubic regression model as it will still fit a little bit of noise with the data it is analyzing. Again the cubic regression model has a benefit over the linear regression model which is flexiblity and flexbility in models normally lead to lower training RSS's.

### (d) Not Linear Testing

When comparing the testing RSS between a linear regression model and a cubic regression model, the bias-variance trade off becomes involved in this. Since the relationship between X and Y is non-linear, a linear regression model would have to deal with a significant amount of bias because the figure can be exponential or a different type of curve. So the testing RSS of a cubic regression model would be lower and better than that of a linear regression model. 

## 3. ISLR 3.7.3 - Extra Practice 

## 3. ISLR 3.7.5 - Extra Practice 